<a href="https://colab.research.google.com/github/Buddha09/Interpretable-Classification-of-Time-series-using-Euler-Characteristic-Surfaces/blob/main/ECS_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interpretable Classification of Time Series Using Euler Characteristic Surfaces

Reference implementation for the manuscript
**"Interpretable Classification of Time Series Using Euler Characteristic Surfaces"**.

This notebook contains the complete pipeline:

1. **Core ECS construction** — Takens delay embedding, Alpha-complex filtration, $K$-window Euler Characteristic Surface.
2. **Classifiers** — AUC-ranked single-feature threshold classifier and an AdaBoost ensemble.
3. **Rössler demonstration** — periodic vs chaotic ECS.
4. **Biomedical classification** — ECG5000, Epilepsy2, TwoLeadECG, ECG200, ECGFiveDays (UCR archive).
5. **Ablation studies** — Part A (component attribution) and Part B (topology vs raw-window features).
6. **Baseline comparison** — 1-NN DTW, Shapelet Transform, ROCKET, MiniROCKET.
7. **Interpretability** — classical time/frequency features in the ECS-selected window.
8. **FNN analysis** — embedding-dimension selection.

All datasets use the standard UCR train/test splits with no additional preprocessing.

> **Repository:** https://github.com/Buddha09/Interpretable-Classification-of-Time-series-using-Euler-Characteristic-Surfaces


## 0. Setup

Install dependencies (run once).

In [ ]:
# !pip install gudhi aeon scikit-learn numpy scipy matplotlib
# Uncomment the line above on first run.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import gudhi
from scipy.integrate import solve_ivp
from scipy.spatial import cKDTree
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

RNG = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("Environment ready.")

## 1. Core ECS functions

The Euler Characteristic Surface is built in three steps:

1. **Takens embedding** — reconstruct a phase-space point cloud from the scalar series.
2. **Alpha-complex filtration** — for a window of points, compute the Euler characteristic
   $\chi(r) = V - E + F - \dots$ as the filtration scale $r$ grows. We use the
   `gudhi` Alpha complex, whose filtration value equals the **squared circumradius**, so a
   simplex enters at scale $r$ when its filtration value $\le r^2$.
3. **$K$-window partitioning** — split the point cloud into $K$ contiguous temporal blocks
   and evaluate the Euler curve over $R$ scales in each block, giving a $K\times R$ matrix.

In [ ]:
def takens_embedding(x, m=3, tau=1):
    """Delay embedding: point i = [x(i), x(i+tau), ..., x(i+(m-1)*tau)]."""
    x = np.asarray(x, dtype=float)
    N = len(x) - (m - 1) * tau
    if N <= 0:
        raise ValueError(f"Series too short: len={len(x)}, m={m}, tau={tau}")
    return np.column_stack([x[j * tau: j * tau + N] for j in range(m)])


def euler_char_curve(points, scales):
    """Euler characteristic of the Alpha complex at each filtration scale.

    gudhi AlphaComplex filtration value = squared circumradius, so a simplex
    enters at scale r when filtration <= r**2.  chi(r) = sum (-1)^dim.
    """
    points = np.asarray(points, dtype=float)
    scales = np.asarray(scales, dtype=float)
    n = len(points)
    if n == 0:
        return np.zeros(len(scales))
    if n == 1:
        return np.ones(len(scales))
    # Degenerate (collinear / identical) clouds break Delaunay; jitter slightly.
    if np.allclose(points.std(axis=0), 0):
        points = points + 1e-9 * np.random.default_rng(0).standard_normal(points.shape)
    try:
        st = gudhi.AlphaComplex(points=points).create_simplex_tree()
    except Exception:
        return np.full(len(scales), float(n))   # only vertices survive
    dims = np.array([len(s) - 1 for s, _ in st.get_filtration()])
    filt = np.array([f for _, f in st.get_filtration()])
    signs = np.where(dims % 2 == 0, 1, -1)
    order = np.argsort(filt)
    fs = filt[order]
    cumsum = np.cumsum(signs[order])
    idx = np.searchsorted(fs, scales ** 2, side="right") - 1
    return np.where(idx >= 0, cumsum[np.clip(idx, 0, len(cumsum) - 1)], 0).astype(float)


def compute_ecs(x, m=3, tau=1, K=10, R=10, scale_max=1.0):
    """K-window Euler Characteristic Surface (returns K x R matrix and scales)."""
    pts = takens_embedding(x, m, tau)
    N = len(pts)
    step = int(np.ceil(N / K))
    scales = np.linspace(1e-9, scale_max, R)
    ecs = np.zeros((K, R))
    for k in range(K):
        s = k * step
        e = N if k == K - 1 else min(s + step, N)
        ecs[k] = euler_char_curve(pts[s:e], scales)
    return ecs, scales


def build_ecs_features(X, **kw):
    """Stack flattened ECS feature vectors for a set of signals X (n, L)."""
    return np.array([compute_ecs(s, **kw)[0].flatten() for s in X])


# quick sanity check on a triangle (hand-verifiable)
_tri = np.array([[0., 0.], [1., 0.], [0.5, 0.8]])
_chi = euler_char_curve(_tri, np.array([0.3, 0.48, 0.51, 0.6]))
print("Triangle EC curve:", _chi.astype(int), "(expected [3 1 0 1])")

## 2. Classifiers

**Single-feature classifier (ECS-SF).** Every ECS cell is a candidate feature.
We rank all $K\times R$ cells by training-set AUC, pick the best cell, choose a polarity
$p\in\{+1,-1\}$, and set a Youden-optimal threshold. The decision is fully traceable to
one $(r^*, k^*)$ coordinate.

**AdaBoost ensemble (ECS-Ada).** Decision stumps over the full ECS feature grid, aggregated
by AdaBoost. The per-feature $\alpha$-weights reveal whether the discriminant is localized
or distributed.

In [ ]:
class ECSSingleFeature:
    """AUC-ranked single ECS feature + polarity + Youden threshold."""

    def fit(self, F, y):
        y = np.asarray(y)
        best = (-np.inf, 0, 1, 0.0)  # (auc, feature_idx, polarity, threshold)
        for j in range(F.shape[1]):
            col = F[:, j]
            if np.std(col) == 0:
                continue
            for p in (1, -1):
                s = p * col
                auc = roc_auc_score(y, s)
                if auc > best[0]:
                    best = (auc, j, p, self._youden(s, y))
        self.auc_, self.j_, self.p_, self.thr_ = best
        return self

    @staticmethod
    def _youden(s, y):
        P = (y == 1).sum(); Nn = (y == 0).sum()
        best_thr, best_j = s.min() - 1, -np.inf
        for c in np.unique(s):
            pred = (s >= c)
            tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum()
            j = (tp / P if P else 0) - (fp / Nn if Nn else 0)
            if j > best_j:
                best_j, best_thr = j, c
        return best_thr

    def decision(self, F):
        return self.p_ * F[:, self.j_]

    def predict(self, F):
        return (self.decision(F) >= self.thr_).astype(int)


class ECSAdaBoost:
    """AdaBoost over decision stumps on the ECS feature grid."""

    def __init__(self, n_estimators=100, seed=42):
        self.clf = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=n_estimators, random_state=seed)

    def fit(self, F, y):
        self.clf.fit(F, np.asarray(y))
        return self

    def predict(self, F):
        return self.clf.predict(F)

    def decision(self, F):
        return self.clf.predict_proba(F)[:, 1]

    def alpha_map(self, K, R):
        """Total AdaBoost weight attributed to each ECS cell (K x R)."""
        w = np.zeros(K * R)
        for est, a in zip(self.clf.estimators_, self.clf.estimator_weights_):
            feat = est.tree_.feature[0]
            if feat >= 0:
                w[feat] += a
        return w.reshape(K, R)


def evaluate(y, yhat, score):
    """Six-metric evaluation: AUC, Accuracy, F1, Precision, Recall, Specificity."""
    y = np.asarray(y)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return dict(AUC=roc_auc_score(y, score), Acc=accuracy_score(y, yhat),
                F1=f1_score(y, yhat, zero_division=0),
                Prec=precision_score(y, yhat, zero_division=0),
                Rec=recall_score(y, yhat, zero_division=0),
                Spec=tn / (tn + fp) if (tn + fp) else 0.0)

print("Classifiers defined.")

## 3. Rössler system: periodic vs chaotic ECS

The Rössler system transitions from periodic ($c=2.3$) to chaotic ($c=7.3$) dynamics.
The ECS of the chaotic regime is rougher and takes higher $\chi$ values, reflecting the
increased topological complexity of the strange attractor.

In [ ]:
def rossler_x(c, n=300, dt=0.5, s0=(1.0, 1.0, 1.0)):
    """x-component of the Rössler system (a=b=0.2)."""
    def f(t, s):
        x, y, z = s
        return [-y - z, x + 0.2 * y, 0.2 + z * (x - c)]
    sol = solve_ivp(f, [0, n * dt], list(s0),
                    t_eval=np.arange(0, n * dt, dt), rtol=1e-8, atol=1e-10)
    return sol.y[0]


x_per = rossler_x(2.3)
x_cha = rossler_x(7.3)
ecs_per, scl = compute_ecs(x_per, m=3, tau=1, K=10, R=10, scale_max=5.0)
ecs_cha, _   = compute_ecs(x_cha, m=3, tau=1, K=10, R=10, scale_max=5.0)
print(f"Periodic ECS mean chi = {ecs_per.mean():.2f}")
print(f"Chaotic  ECS mean chi = {ecs_cha.mean():.2f}")

fig, ax = plt.subplots(2, 2, figsize=(11, 6))
ax[0, 0].plot(x_per, lw=0.8, color="#2166ac"); ax[0, 0].set_title("(a) Periodic (c=2.3)")
ax[0, 1].plot(x_cha, lw=0.8, color="#d6604d"); ax[0, 1].set_title("(b) Chaotic (c=7.3)")
im0 = ax[1, 0].imshow(ecs_per, aspect="auto", origin="lower", cmap="viridis")
ax[1, 0].set_title("(c) ECS periodic"); ax[1, 0].set_xlabel("scale index r"); ax[1, 0].set_ylabel("window k")
im1 = ax[1, 1].imshow(ecs_cha, aspect="auto", origin="lower", cmap="viridis")
ax[1, 1].set_title("(d) ECS chaotic"); ax[1, 1].set_xlabel("scale index r"); ax[1, 1].set_ylabel("window k")
fig.colorbar(im0, ax=ax[1, 0]); fig.colorbar(im1, ax=ax[1, 1])
plt.tight_layout(); plt.show()

## 4. Loading UCR datasets

We use the `aeon` archive loader. Each dataset is loaded with its standard train/test split.
`load_classification` returns `X` of shape `(n_cases, n_channels, n_timepoints)`; for these
univariate signals `n_channels = 1`.

The `positive_class` argument defines which label is treated as class 1; for **ECG5000** we
keep the two most populous classes (`"1"` normal, `"2"` ectopic) following the binary protocol
in the literature. For **Epilepsy2** the seizure class is positive.

In [ ]:
from aeon.datasets import load_classification

# Per-dataset configuration used in the paper.
# m: embedding dimension (FNN-selected); K, R: ECS grid; scale_max: max filtration scale.
# pos: label(s) mapped to class 1; neg: label(s) mapped to class 0 (None = "all others").
DATASETS = {
    "ECG5000":     dict(m=3, K=10, R=10, scale_max=1.0,  pos=["1"], neg=["2"]),
    "Epilepsy2":   dict(m=4, K=9,  R=9,  scale_max=1.0,  pos=["1"], neg=None),
    "TwoLeadECG":  dict(m=3, K=8,  R=8,  scale_max=1.0,  pos=["1"], neg=None),
    "ECG200":      dict(m=3, K=8,  R=8,  scale_max=1.0,  pos=["1"], neg=None),
    "ECGFiveDays": dict(m=3, K=8,  R=8,  scale_max=1.0,  pos=["1"], neg=None),
}


def load_binary(name, pos, neg=None):
    """Load a UCR dataset and map labels to {0,1}."""
    Xtr, ytr = load_classification(name, split="train")
    Xte, yte = load_classification(name, split="test")
    Xtr = np.asarray(Xtr)[:, 0, :]
    Xte = np.asarray(Xte)[:, 0, :]
    ytr = np.asarray(ytr).astype(str)
    yte = np.asarray(yte).astype(str)
    pos = set(pos)

    def remap(X, y):
        if neg is None:               # positive vs all-others
            yb = np.isin(y, list(pos)).astype(int)
            return X, yb
        keep = np.isin(y, list(pos) + list(neg))   # binary subset
        Xk, yk = X[keep], y[keep]
        return Xk, np.isin(yk, list(pos)).astype(int)

    Xtr, ytr = remap(Xtr, ytr)
    Xte, yte = remap(Xte, yte)
    return Xtr, ytr, Xte, yte


# Example (requires internet to download from the UCR archive):
# Xtr, ytr, Xte, yte = load_binary("ECG5000", **{k: DATASETS["ECG5000"][k] for k in ("pos", "neg")})
# print("ECG5000:", Xtr.shape, Xte.shape, "class balance:", ytr.mean().round(3))

## 5. Main classification driver

For each dataset: build ECS features for train and test, fit both classifiers, and report
all six metrics. The single-feature classifier additionally exposes the selected
$(r^*, k^*)$ coordinate.

In [ ]:
def run_dataset(name, n_ada=None, verbose=True):
    cfg = DATASETS[name]
    Xtr, ytr, Xte, yte = load_binary(name, cfg["pos"], cfg["neg"])
    kw = dict(m=cfg["m"], tau=1, K=cfg["K"], R=cfg["R"], scale_max=cfg["scale_max"])
    Ftr = build_ecs_features(Xtr, **kw)
    Fte = build_ecs_features(Xte, **kw)

    sf = ECSSingleFeature().fit(Ftr, ytr)
    m_sf = evaluate(yte, sf.predict(Fte), sf.decision(Fte))
    k_star, r_star_idx = divmod(sf.j_, cfg["R"])

    n_ada = n_ada or cfg["K"] * cfg["R"]
    ada = ECSAdaBoost(n_estimators=n_ada).fit(Ftr, ytr)
    m_ada = evaluate(yte, ada.predict(Fte), ada.decision(Fte))

    if verbose:
        print(f"\n=== {name} ===")
        print(f"  selected coordinate: window k*={k_star} (0-indexed), scale index={r_star_idx}, polarity={sf.p_}")
        print(f"  ECS-SF : " + "  ".join(f"{k}={v:.3f}" for k, v in m_sf.items()))
        print(f"  ECS-Ada: " + "  ".join(f"{k}={v:.3f}" for k, v in m_ada.items()))
    return dict(name=name, sf=sf, ada=ada, m_sf=m_sf, m_ada=m_ada,
                Ftr=Ftr, Fte=Fte, Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, cfg=cfg)


# Run all five datasets (uncomment once data can be downloaded):
# results = {name: run_dataset(name) for name in DATASETS}

## 6. Ablation Part A — component attribution

Compare three configurations of increasing complexity:
**ECC ($K=1$)** — global Euler curve, no temporal partitioning;
**ECS-SF** — single best ECS feature;
**ECS-Ada** — full AdaBoost ensemble.
The $K$-window gain measures how localized the discriminant is; the AdaBoost gain measures
how distributed it is.

In [ ]:
def ablation_part_A(name):
    cfg = DATASETS[name]
    Xtr, ytr, Xte, yte = load_binary(name, cfg["pos"], cfg["neg"])

    # (i) ECC: K=1 global Euler curve
    ke = dict(m=cfg["m"], tau=1, K=1, R=cfg["R"], scale_max=cfg["scale_max"])
    Ecc_tr = build_ecs_features(Xtr, **ke); Ecc_te = build_ecs_features(Xte, **ke)
    sf_ecc = ECSSingleFeature().fit(Ecc_tr, ytr)
    m_ecc = evaluate(yte, sf_ecc.predict(Ecc_te), sf_ecc.decision(Ecc_te))

    # (ii)+(iii) ECS-SF and ECS-Ada
    kw = dict(m=cfg["m"], tau=1, K=cfg["K"], R=cfg["R"], scale_max=cfg["scale_max"])
    Ftr = build_ecs_features(Xtr, **kw); Fte = build_ecs_features(Xte, **kw)
    sf = ECSSingleFeature().fit(Ftr, ytr)
    m_sf = evaluate(yte, sf.predict(Fte), sf.decision(Fte))
    ada = ECSAdaBoost(n_estimators=cfg["K"] * cfg["R"]).fit(Ftr, ytr)
    m_ada = evaluate(yte, ada.predict(Fte), ada.decision(Fte))

    print(f"\n=== Part A: {name} ===")
    for label, mtr in [("ECC (K=1)", m_ecc), ("ECS-SF", m_sf), ("ECS-Ada", m_ada)]:
        print(f"  {label:10s}: " + "  ".join(f"{k}={v:.3f}" for k, v in mtr.items()))
    return m_ecc, m_sf, m_ada


# for name in DATASETS: ablation_part_A(name)

## 7. Ablation Part B — topology vs raw-window features

Fix the window $k^*$ and scale $r^*$ chosen by the ECS, and compare the Euler-characteristic
feature against features extracted from the **same raw signal window**:
**B1** mean, **B2** std, **B3** range, **B4** their AdaBoost combination,
**B5** a class-template $\ell_2$ distance.
Because B1–B5 are handed the ECS-discovered window, any residual advantage of the topological
feature is a conservative lower bound on its added value.

For $\tau=1$ the $i$-th embedded point starts at raw index $i$, so the point-cloud window maps
directly to a contiguous raw-signal window.

In [ ]:
def window_to_raw(L, m, tau, K, k0):
    """Map 0-indexed point-cloud window k0 to raw signal indices (valid for tau=1)."""
    N = L - (m - 1) * tau
    step = int(np.ceil(N / K))
    pc_start = k0 * step
    pc_end = N if k0 == K - 1 else min(pc_start + step, N)
    return pc_start, pc_end + (m - 1) * tau   # include all dims of last point


def raw_window_features(X, a, b):
    seg = X[:, a:b]
    return dict(B1_mean=seg.mean(1), B2_std=seg.std(1),
                B3_range=seg.max(1) - seg.min(1))


def template_distance(Xtr, ytr, Xte, a, b):
    s_tr, s_te = Xtr[:, a:b], Xte[:, a:b]
    t0 = s_tr[ytr == 0].mean(0); t1 = s_tr[ytr == 1].mean(0)
    return np.linalg.norm(s_te - t0, axis=1) - np.linalg.norm(s_te - t1, axis=1)


def ablation_part_B(res):
    """res: output dict from run_dataset (provides the fitted ECS-SF and data)."""
    cfg = res["cfg"]; sf = res["sf"]
    Xtr, ytr, Xte, yte = res["Xtr"], res["ytr"], res["Xte"], res["yte"]
    L = Xtr.shape[1]
    k0 = sf.j_ // cfg["R"]
    a, b = window_to_raw(L, cfg["m"], 1, cfg["K"], k0)
    print(f"\n=== Part B: {res['name']} (window k*={k0} -> raw [{a}:{b}]) ===")

    rf_te = raw_window_features(Xte, a, b)
    rows = {}
    for nm, sc in rf_te.items():
        rows[nm] = roc_auc_score(yte, sc)
    rows["B5_template"] = roc_auc_score(yte, template_distance(Xtr, ytr, Xte, a, b))
    rows["B6_ECS"] = res["m_sf"]["AUC"]
    rows["B7_ECS_Ada"] = res["m_ada"]["AUC"]
    for nm, auc in rows.items():
        print(f"  {nm:13s} AUC={auc:.3f}")
    return rows


# for name in DATASETS: ablation_part_B(results[name])

## 8. Baseline comparison

Four representative TSC methods under the identical train/test protocol:
**1-NN DTW**, **Shapelet Transform**, **ROCKET**, **MiniROCKET**.

In [ ]:
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
from aeon.classification.convolution_based import RocketClassifier, MiniRocketClassifier
from aeon.classification.shapelet_based import ShapeletTransformClassifier


def run_baselines(name, n_kernels=10000):
    cfg = DATASETS[name]
    Xtr, ytr, Xte, yte = load_binary(name, cfg["pos"], cfg["neg"])
    Xtr3 = Xtr[:, None, :]; Xte3 = Xte[:, None, :]
    ytr_s = ytr.astype(str); yte_s = yte.astype(str)

    models = {
        "1NN-DTW": KNeighborsTimeSeriesClassifier(distance="dtw", n_neighbors=1),
        "STC": ShapeletTransformClassifier(n_shapelet_samples=500, max_shapelets=100,
                                           batch_size=100),
        "ROCKET": RocketClassifier(n_kernels=n_kernels),
        "MiniROCKET": MiniRocketClassifier(n_kernels=n_kernels),
    }
    print(f"\n=== Baselines: {name} ===")
    out = {}
    for nm, clf in models.items():
        clf.fit(Xtr3, ytr_s)
        yhat = clf.predict(Xte3)
        try:
            proba = clf.predict_proba(Xte3)[:, list(clf.classes_).index("1")]
            auc = roc_auc_score((yte_s == "1").astype(int), proba)
        except Exception:
            auc = roc_auc_score((yte_s == "1").astype(int), (yhat == "1").astype(int))
        acc = (yhat == yte_s).mean()
        out[nm] = dict(AUC=auc, Acc=acc)
        print(f"  {nm:11s} AUC={auc:.4f}  Acc={acc:.4f}")
    return out


# for name in DATASETS: run_baselines(name)

## 9. Interpretability — classical features in the ECS window

For the ECS-selected window we extract standard time-domain statistics and the dominant
frequency (zero-padded DFT). These connect the topological feature to the amplitude and
frequency differences that classical ECG classifiers exploit.

In [ ]:
def classical_and_psd(seg, nfft=512):
    """Time-domain stats + dominant frequency for each row (heartbeat) in seg."""
    freqs = np.fft.rfftfreq(nfft)
    P = np.abs(np.fft.rfft(seg, n=nfft, axis=1)) ** 2 / nfft
    return dict(mean=seg.mean(1), std=seg.std(1),
                rng=seg.max(1) - seg.min(1),
                f_dom=freqs[P.argmax(1)])


def interpretability_table(res):
    cfg = res["cfg"]; sf = res["sf"]
    Xte, yte = res["Xte"], res["yte"]
    L = Xte.shape[1]; k0 = sf.j_ // cfg["R"]
    a, b = window_to_raw(L, cfg["m"], 1, cfg["K"], k0)
    seg = Xte[:, a:b]
    f1 = classical_and_psd(seg[yte == 1]); f0 = classical_and_psd(seg[yte == 0])
    print(f"\n=== Classical features in window [{a}:{b}] for {res['name']} ===")
    print(f"  {'feature':10s} {'class1':>10s} {'class0':>10s} {'ratio':>8s}")
    for key in ["mean", "std", "rng", "f_dom"]:
        c1 = float(np.mean(f1[key])); c0 = float(np.mean(f0[key]))
        ratio = c1 / c0 if c0 else np.inf
        print(f"  {key:10s} {c1:10.4f} {c0:10.4f} {ratio:8.2f}")


# interpretability_table(results["ECG5000"])

## 10. False Nearest Neighbours (embedding dimension)

The FNN ratio is computed at $\tau=1$ for $m=1,\dots,6$. The optimal embedding dimension is
the smallest $m$ at which the ratio drops below 1%.

In [ ]:
def fnn_ratio(x, m, tau=1, Rtol=15.0):
    """Fraction of false nearest neighbours when going from dimension m to m+1."""
    em = takens_embedding(x, m, tau)
    e1 = takens_embedding(x, m + 1, tau)
    n = len(e1); em = em[:n]
    d, idx = cKDTree(em).query(em, k=2)
    nn_d, nn_i = d[:, 1], idx[:, 1]
    false = valid = 0
    for i in range(n):
        if nn_d[i] == 0:
            continue
        valid += 1
        if abs(e1[i, -1] - e1[nn_i[i], -1]) / nn_d[i] > Rtol:
            false += 1
    return false / valid if valid else 0.0


def fnn_curve(X, m_max=6, tau=1):
    """Mean FNN ratio across a set of signals, for m = 1..m_max."""
    ratios = np.zeros(m_max)
    for m in range(1, m_max + 1):
        ratios[m - 1] = np.mean([fnn_ratio(sig, m, tau) for sig in X])
    return ratios


# Demonstration on a clean sinusoid:
x_demo = np.sin(np.linspace(0, 20 * np.pi, 178))
r_demo = [fnn_ratio(x_demo, m) for m in range(1, 7)]
print("FNN ratios (sinusoid) m=1..6:", [round(v, 3) for v in r_demo])

# For a dataset (uncomment when data available):
# ratios = fnn_curve(results["ECG5000"]["Xtr"], m_max=6)
# plt.plot(range(1, 7), ratios, "o-"); plt.axhline(0.01, ls="--", c="gray")
# plt.xlabel("embedding dimension m"); plt.ylabel("FNN ratio"); plt.show()

## 11. Illustration — Euler characteristic vs point-cloud density

A spread-out trajectory keeps $\chi$ near its maximum (few edges); a dense cluster fills in
edges and triangles, driving $\chi$ down. This is the geometric mechanism behind the ECS
class separation.

In [ ]:
from matplotlib.patches import Circle, Polygon

fig, ax = plt.subplots(1, 2, figsize=(11, 5)); r_star = 0.45

# left: spread-out -> chi = V
pts_s = np.array([[0.5, 0.5], [3.5, 0.65], [3.05, 2.55], [1.1, 2.85], [2.0, 1.65]])
for p in pts_s:
    ax[0].add_patch(Circle(p, r_star, fill=True, fc="#aec6e8", alpha=0.25, ec="#2166ac", ls="--"))
    ax[0].plot(*p, "o", color="#2166ac", ms=11)
ax[0].set_title(r"Spread-out: $\chi = V = 5$"); ax[0].set_xlim(-.3, 4.3); ax[0].set_ylim(-.3, 3.6)
ax[0].set_aspect("equal"); ax[0].axis("off")

# right: dense cluster -> chi = 1 (V=4, E=6, F=3)
p0, p1, p2, p3 = (2.0, 1.3), (2.6, 1.3), (2.3, 1.9), (2.0, 1.9)
for tri in [(p0, p1, p2), (p0, p1, p3), (p0, p2, p3)]:
    ax[1].add_patch(Polygon(tri, fc="#d6604d", alpha=0.25, ec="none"))
for p in [p0, p1, p2, p3]:
    ax[1].add_patch(Circle(p, r_star, fill=True, fc="#f4c2b0", alpha=0.20, ec="#d6604d", ls="--"))
for a_, b_ in [(p0, p1), (p0, p2), (p0, p3), (p1, p2), (p1, p3), (p2, p3)]:
    ax[1].plot([a_[0], b_[0]], [a_[1], b_[1]], color="#d6604d", lw=2)
for p in [p0, p1, p2, p3]:
    ax[1].plot(*p, "o", color="#d6604d", ms=11)
ax[1].set_title(r"Dense: $\chi = V-E+F = 4-6+3 = 1$"); ax[1].set_xlim(-.3, 4.3); ax[1].set_ylim(-.3, 3.6)
ax[1].set_aspect("equal"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## Reproducing the paper

To reproduce all results, ensure an internet connection (the UCR archive is downloaded on
first use) and run:

```python
results = {name: run_dataset(name) for name in DATASETS}      # main classification
for name in DATASETS: ablation_part_A(name)                   # ablation A
for name in DATASETS: ablation_part_B(results[name])          # ablation B
for name in DATASETS: run_baselines(name)                     # baselines
interpretability_table(results["ECG5000"])                    # interpretability
```

Grid sizes (`K`, `R`) in `DATASETS` are the cross-validation-selected values reported in the
manuscript; the embedding dimensions `m` are FNN-selected ($m=4$ for Epilepsy2, $m=3$ for the
ECG datasets).